# QLoRA-train a Qwen student on research traces (Kaggle free GPU)

Thin wrapper around `train/train_qlora.py` from
[Fine-Tune-Models](https://github.com/ashcastelinocs124/Fine-Tune-Models).
Works for any domain profile (macro / general_search) — set MODEL and the
dataset name in the config cell.

**Before running:**
1. **Settings → Accelerator**: `GPU T4 x2` or `GPU P100` (script uses one GPU).
2. **Settings → Internet**: ON (clones the repo, downloads the base model).
3. **Add Input → your private Kaggle Dataset** containing the clean traces
   (`*.json` from the repo pipeline). Set its name in the config cell below.
4. *(optional, for the HF push)* **Add-ons → Secrets**: add `HF_TOKEN` with a
   Hugging Face write token, and set `HF_REPO` in the config cell.

Output: LoRA adapter zipped at `/kaggle/working/{OUT_NAME}.zip` — download it
from the Output tab — and (if configured) pushed to the HF repo.

In [ ]:
# ---- Config ----
TRACES_DATASET = "search-traces-clean"   # name of the attached Kaggle Dataset
MODEL = "Qwen/Qwen2.5-7B-Instruct"       # base model to fine-tune
OUT_NAME = "qwen25-7b-search-lora"       # output dir / zip / HF folder name
HF_REPO = "ashcash15/qwen2.5-7b-search"  # HF model repo to push the adapter to ("" = skip push)
MAX_LEN = 8192   # 16384 OOMs on 16GB T4/P100 with a 7-8B model; raise on bigger GPUs
EPOCHS = 2
LR = 2e-4

In [ ]:
%cd /kaggle/working
!rm -rf Fine-Tune-Models
!git clone --depth 1 https://github.com/ashcastelinocs124/Fine-Tune-Models.git
%cd Fine-Tune-Models
# pyproject pins transformers<5; peft/bitsandbytes/accelerate are the GPU extras
!pip install -q -e . peft bitsandbytes accelerate

In [ ]:
# ---- Stage traces from the attached Kaggle Dataset ----
import glob, os, shutil

os.makedirs("data/traces/clean", exist_ok=True)
src = f"/kaggle/input/{TRACES_DATASET}"
files = glob.glob(f"{src}/**/*.json", recursive=True)
if not files:
    attached = os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else []
    inside = glob.glob(f"{src}/**/*", recursive=True)[:20]
    raise SystemExit(
        f"no .json files under {src}\n"
        f"datasets attached to this notebook: {attached or 'NONE — click Add Input and attach your dataset'}\n"
        f"first files inside it: {inside or '(dataset not attached, or empty)'}"
    )
for f in files:
    shutil.copy(f, "data/traces/clean/")
print(f"staged {len(files)} traces")

In [ ]:
!python train/train_qlora.py --model {MODEL} --clean data/traces/clean --out outputs/{OUT_NAME} \
    --epochs {EPOCHS} --lr {LR} --max-len {MAX_LEN}

In [ ]:
# ---- Package the adapter for download ----
import shutil

shutil.make_archive(f"/kaggle/working/{OUT_NAME}", "zip", f"outputs/{OUT_NAME}")
print(f"adapter at /kaggle/working/{OUT_NAME}.zip")

In [ ]:
# ---- Push the adapter to the HF Hub (skipped unless HF_REPO is set) ----
if HF_REPO:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi

    token = UserSecretsClient().get_secret("HF_TOKEN")  # Add-ons -> Secrets
    HfApi(token=token).upload_folder(
        repo_id=HF_REPO,
        folder_path=f"outputs/{OUT_NAME}",
        commit_message=f"LoRA adapter: {OUT_NAME}",
    )
    print(f"pushed adapter to https://huggingface.co/{HF_REPO}")
else:
    print("HF_REPO empty - skipping push")